## 00. Imports & Config

In [3]:
import os
import sys
import importlib
import pandas as pd 
import warnings

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


from api_retrieval.markets.exus.database_extractor import DatabaseExtractor

warnings.filterwarnings('ignore')

db = DatabaseExtractor(
    server="exusbigcloud.database.windows.net",
    username="exusbigcloud_readonly",
    password="H8blZuJ0tju2"
)

In [4]:
quarter = '26Q1'

In [5]:
import pyodbc
print(pyodbc.drivers())


['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


## 01. Extract DAM prices data

In [6]:
print("EuropeDAMActuals columns:")
print(db.list_columns("bigcloud_db", "EuropeDAMActuals"))

EuropeDAMActuals columns:
['CountryCode', 'PriceAreaCode', 'Resolution', 'Unit', 'Price', 'Market', 'UTCTime', 'LocalTime', 'id']


In [7]:
countries = ['DE', 'IT', 'ES', 'FR', 'PT', 'PL', 'GB']

df_DAM = db.query_table(
    "bigcloud_db",
    "dbo.EuropeDAMActuals",
    order_by="UTCTime DESC",
    filters={"CountryCode": countries}
)

In [8]:
df_DAM_mod = df_DAM.copy()

# To standarize time for quarterhourly data and hourly data, we will group data by hour and take the mean of the quarterhourly data. This way, we will have a single value for each hour, which will be comparable to the hourly data.

# Parse LocalTime as UTC-aware datetime and drop the tz component before bucketing
df_DAM_mod["LocalTime"] = pd.to_datetime(df_DAM_mod["LocalTime"],
    format='ISO8601',
    utc=True
)

df_DAM_mod['LocalDate'] = df_DAM_mod['LocalTime'].dt.date
df_DAM_mod["LocalHour"] = df_DAM_mod["LocalTime"].dt.hour

df_DAM_mod['Month'] = df_DAM_mod['LocalTime'].dt.to_period('M')
df_DAM_mod['Year'] = df_DAM_mod['LocalTime'].dt.year
 
# Average to hourly
df_hour = (
    df_DAM_mod
    .groupby(["CountryCode", "PriceAreaCode", "Unit", "Market", "LocalDate", "LocalHour"], as_index=False)["Price"]
    .mean()
)

In [9]:
# Now get the average monthly prices for each country and price area. This will be useful for the analysis of the price trends over time.

df_hour['LocalDate'] = pd.to_datetime(df_hour['LocalDate'])
df_hour['Month'] = df_hour['LocalDate'].dt.month
df_hour['Year'] = df_hour['LocalDate'].dt.year

df_monthly = (
    df_hour
    .groupby(["CountryCode", "PriceAreaCode", "Unit", "Market", "Year", "Month"], as_index=False)["Price"]
    .mean()
)

In [13]:
df_hour[(df_hour['Year'] == 2026) & (df_hour['Price'] < 0) & (df_hour['CountryCode'] == 'GB')] 
df_hour['CountryCode'].unique()

array(['DE', 'ES', 'FR', 'IT', 'PL', 'PT'], dtype=object)

In [11]:
start_date = "2025-04-01"
end_date = "2026-03-31"
df_filtered = df_monthly[(df_monthly['Year'] >= int(start_date[:4])) & (df_monthly['Year'] <= int(end_date[:4]))].sort_values(by=['CountryCode', 'PriceAreaCode', 'Year', 'Month'])
######### cambiar filtrode 2025 al ultimo quarterly, ver proyecto bloomberg #########

df_pivot = df_filtered.pivot_table(columns=['Year', 'Month'], index=['CountryCode', 'PriceAreaCode', 'Market', 'Unit'], values='Price')

df_pivot.to_excel(f"monthly_prices_{quarter}.xlsx")

df_pivot

Year                                                   2025              \
Month                                                    1           2    
CountryCode PriceAreaCode   Market      Unit                              
DE          DE-LU           DAM         €/MWh    114.321304  128.509360   
ES          ES              DAM         €/MWh     96.693454  108.260476   
FR          FR              DAM         €/MWh    102.442473  122.630298   
IT          IT-Calabria     DAM         €/MWh    141.295699  148.901161   
            IT-Centre-North DAM         €/MWh    143.571868  150.648289   
            IT-Centre-South DAM         €/MWh    143.131895  150.020417   
            IT-North        DAM         €/MWh    143.324476  150.536012   
            IT-SACOAC       DAM         €/MWh    139.639879  146.271220   
            IT-SACODC       DAM         €/MWh    143.572352  169.771875   
            IT-Sardinia     DAM         €/MWh    139.639879  146.271220   
            IT-Sicily       DAM         €/MWh    141.387957  151.092202   
            IT-South        DAM         €/MWh    142.171546  150.020417   
PL          PL              DAM         €/MWh    114.225309  133.812336   
                            PL FIXING I PLN/MWh  479.448669  562.636161   
PT          PT              DAM         €/MWh     96.735067  108.168631   

Year                                                                     \
Month                                                    3           4    
CountryCode PriceAreaCode   Market      Unit                              
DE          DE-LU           DAM         €/MWh     94.692097   77.925528   
ES          ES              DAM         €/MWh     53.104355   26.650917   
FR          FR              DAM         €/MWh     76.839126   42.052514   
IT          IT-Calabria     DAM         €/MWh    118.459758   99.172722   
            IT-Centre-North DAM         €/MWh    121.241277  100.363153   
            IT-Centre-South DAM         €/MWh    119.600255   99.760819   
            IT-North        DAM         €/MWh    121.179315  100.266556   
            IT-SACOAC       DAM         €/MWh    117.194220   92.575639   
            IT-SACODC       DAM         €/MWh    121.241277   99.541069   
            IT-Sardinia     DAM         €/MWh    117.194220   92.575639   
            IT-Sicily       DAM         €/MWh    118.730000   99.157333   
            IT-South        DAM         €/MWh    118.459758   99.172722   
PL          PL              DAM         €/MWh     99.264879   85.964736   
                            PL FIXING I PLN/MWh  415.267298  369.198514   
PT          PT              DAM         €/MWh     52.608172   25.748208   

Year                                                                     \
Month                                                    5           6    
CountryCode PriceAreaCode   Market      Unit                              
DE          DE-LU           DAM         €/MWh     67.323118   64.027528   
ES          ES              DAM         €/MWh     16.986949   72.808639   
FR          FR              DAM         €/MWh     19.435618   40.895861   
IT          IT-Calabria     DAM         €/MWh     93.270228  113.944181   
            IT-Centre-North DAM         €/MWh     94.461331  111.634208   
            IT-Centre-South DAM         €/MWh     93.210538  114.268444   
            IT-North        DAM         €/MWh     93.856075  110.265833   
            IT-SACOAC       DAM         €/MWh     88.640995  113.344486   
            IT-SACODC       DAM         €/MWh     94.702110  114.268444   
            IT-Sardinia     DAM         €/MWh     88.640995  113.344486   
            IT-Sicily       DAM         €/MWh     93.627917  114.366389   
            IT-South        DAM         €/MWh     93.175753  113.938264   
PL          PL              DAM         €/MWh     94.043051   81.523847   
                            PL FIXING I PLN/MWh  401.110605  346.123014   
PT          PT         